# Chapter 24 — Gradient, Curl and Divergence

Companion notebook to Chapter 24 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Forms are arrays, as in the notebooks of Chapters 22 and 23. The classical
side of each identity (dot products with unit normals, areas, lengths) is
computed separately from the forms side, so that the two can be compared.

In [1]:
import itertools
import math
import warnings

import numpy as np
import torch
from torch.func import jacrev, vmap

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor


def sgn(perm):
    n_inv = sum(1 for i in range(len(perm)) for j in range(i + 1, len(perm)) if perm[i] > perm[j])
    return -1 if n_inv % 2 else 1


def antisym(t):
    k = t.dim()
    if k < 2:
        return t.clone()
    return sum(sgn(p) * t.permute(*p) for p in itertools.permutations(range(k)))


def wedge(a, b):
    return antisym(torch.tensordot(a, b, dims=0)) / (math.factorial(a.dim()) * math.factorial(b.dim()))


def dx(i, n):
    v = torch.zeros(n)
    v[i] = 1.0
    return v


def d(omega):
    def d_omega(x):
        J = jacrev(omega)(x)
        k = J.dim() - 1
        Tn = J.movedim(-1, 0)
        return sum((-1) ** j * Tn.movedim(0, j) for j in range(k + 1))
    return d_omega


def pullback(F, omega):
    def pb(x):
        J = jacrev(F)(x)
        w = omega(F(x))
        for _ in range(w.dim()):
            w = torch.tensordot(w, J, dims=([0], [0]))
        return w
    return pb


def interior(v, w):
    return torch.tensordot(v, w, dims=([0], [0]))


def gauss(a, b, n):
    x, w = np.polynomial.legendre.leggauss(n)
    return T(0.5 * (b - a) * x + 0.5 * (a + b)), T(0.5 * (b - a) * w)


def periodic(n, length=2 * math.pi):
    return torch.arange(n) * (length / n), torch.full((n,), length / n)


def integrate(component, rules):
    # integral of a function of u over a box, with one (nodes, weights) rule per variable
    grids = torch.meshgrid(*[r[0] for r in rules], indexing="ij")
    weights = torch.meshgrid(*[r[1] for r in rules], indexing="ij")
    U = torch.stack([g.reshape(-1) for g in grids], dim=1)
    W = torch.stack([w.reshape(-1) for w in weights], dim=1).prod(dim=1)
    return (vmap(component)(U) * W).sum()


def top(form, param):
    # the component of the pull-back of a k-form by a parametrization of a k-dimensional piece
    def comp(u):
        w = pullback(param, form)(u)
        return w[tuple(range(w.dim()))]
    return comp


e2 = [dx(0, 2), dx(1, 2)]
e3 = [dx(i, 3) for i in range(3)]
mu3 = wedge(wedge(e3[0], e3[1]), e3[2])

2.15.0+cu132


## 1. The flux form is the star of the work form (Proposition 24.2)

For a general inner product with Gram matrix $G$ on $\mathbb R^n$ and the
canonical orientation, $\mathrm{vol} = \sqrt{\det G}\,dx_1\wedge\dots\wedge
dx_n$. The defining property of the star,
$\beta\wedge\star(Rv) = \langle\beta, Rv\rangle\,\mathrm{vol}$, is checked
for $\star(Rv) = i_v\mathrm{vol}$, with $\langle\beta, Rv\rangle = \beta(v)$.

In [2]:
from torch.func import jvp, hessian

for n in (3, 4):
    Bm = torch.randn(n, n)
    G = Bm @ Bm.T + torch.eye(n)
    mu = dx(0, n)
    for i in range(1, n):
        mu = wedge(mu, dx(i, n))
    vol = torch.sqrt(torch.linalg.det(G)) * mu
    v, beta = torch.randn(n), torch.randn(n)
    Rv = G @ v                                         # components of the covector <v, .>
    flux = interior(v, vol)
    lhs = wedge(beta, flux)
    assert torch.allclose(lhs, (beta @ v) * vol)
    # <beta, Rv> for the inner product of covectors, G^-1
    assert torch.allclose(beta @ torch.linalg.inv(G) @ Rv, beta @ v)
print("beta ^ i_v vol = beta(v) vol, in dimensions 3 and 4 with a random inner product")

beta ^ i_v vol = beta(v) vol, in dimensions 3 and 4 with a random inner product


## 2. The three operators (Definition 24.4, Theorem 24.5, Proposition 24.6)

Gradient, curl and divergence are computed from the forms, with the
exterior derivative of the notebook of Chapter 22, and compared with the
classical formulas.

In [3]:
X = lambda q: torch.stack([q[1] * q[2] ** 2 + q[0], torch.cos(q[0]) * q[2], q[0] * q[1] * q[2] + q[1] ** 2])
f = lambda q: torch.sin(q[0] * q[1]) + q[2] ** 3 * q[0]
p = T([0.7, -1.3, 0.4])
J = jacrev(X)(p)

# divergence: d(i_X vol) = div X vol
flux_form = lambda q: interior(X(q), mu3)
div_form = d(flux_form)(p)[0, 1, 2]
assert torch.allclose(div_form, torch.trace(J))

# curl: i_{curl X} vol = d(RX); in Cartesian coordinates RX has the components of X
dRX = d(X)(p)
curl = T([J[2, 1] - J[1, 2], J[0, 2] - J[2, 0], J[1, 0] - J[0, 1]])
assert torch.allclose(interior(curl, mu3), dRX)
print("div X =", div_form.item(), "  curl X =", curl.tolist())

# identities
grad_f = lambda q: torch.func.grad(f)(q)
curl_of = lambda Y: (lambda q: (lambda Jq: torch.stack([Jq[2, 1] - Jq[1, 2], Jq[0, 2] - Jq[2, 0], Jq[1, 0] - Jq[0, 1]]))(jacrev(Y)(q)))
div_of = lambda Y: (lambda q: torch.trace(jacrev(Y)(q)))
assert torch.allclose(curl_of(grad_f)(p), torch.zeros(3), atol=1e-12)
assert abs(div_of(curl_of(X))(p)) < 1e-12
fX = lambda q: f(q) * X(q)
assert torch.allclose(div_of(fX)(p), f(p) * div_of(X)(p) + grad_f(p) @ X(p))
print("curl grad = 0, div curl = 0, div(fX) = f div X + df(X)")

# potentials on a star-shaped domain (the Poincare lemma)
nodes, weights = np.polynomial.legendre.leggauss(40)
tq, wq = T(0.5 * (nodes + 1.0)), T(0.5 * weights)


def K(form, p0):
    def K_form(xx):
        h = xx - p0
        k = form(p0).dim()
        return sum(wk * tk ** (k - 1) * interior(h, form(p0 + tk * h)) for tk, wk in zip(tq, wq))
    return K_form


origin = torch.zeros(3)
Y = curl_of(X)                                         # a field with div Y = 0
alpha = K(lambda q: interior(Y(q), mu3), origin)       # a 1-form with d alpha = flux form of Y
assert torch.allclose(curl_of(alpha)(p), Y(p))
pot = K(grad_f, origin)                                # a function with gradient grad f
assert torch.allclose(torch.func.grad(pot)(p), grad_f(p))
print("a field without divergence is the curl of K(flux form); a field without curl is the gradient of K(work form)")

# Exercise 24.1
Xe = lambda q: torch.stack([q[1] * q[2], q[0] * q[2], q[0] * q[1]])
Ye = lambda q: torch.stack([-q[1], q[0], q[2]])
Ze = lambda q: torch.stack([q[0] * q[1] ** 2, q[1] * q[2] ** 2, q[2] * q[0] ** 2])
assert torch.allclose(curl_of(Xe)(p), torch.zeros(3)) and div_of(Xe)(p) == 0
assert torch.allclose(curl_of(Ye)(p), T([0.0, 0.0, 2.0])) and div_of(Ye)(p) == 1
assert torch.allclose(curl_of(Ze)(p), -2 * T([p[1] * p[2], p[0] * p[2], p[0] * p[1]]))

# the three fields of Figure 24.2
plane_curl = lambda Y: (lambda q: (lambda Jq: Jq[1, 0] - Jq[0, 1])(jacrev(Y)(q)))
q2 = T([0.3, -0.8])
for name, field, dv, cv in (("(x, y)", lambda q: q, 2.0, 0.0), ("(-y, x)", lambda q: torch.stack([-q[1], q[0]]), 0.0, 2.0),
                            ("(y, 0)", lambda q: torch.stack([q[1], 0 * q[0]]), 0.0, -1.0)):
    assert div_of(field)(q2) == dv and plane_curl(field)(q2) == cv

div X = 0.09000000000000008   curl X = [-3.084842187284489, -0.52, -0.41768707489507645]
curl grad = 0, div curl = 0, div(fX) = f div X + df(X)
a field without divergence is the curl of K(flux form); a field without curl is the gradient of K(work form)


## 3. The divergence for a volume (Proposition 24.8, Example 24.9)

In polar coordinates $\mathrm{vol} = r\,dr\wedge d\theta$. A field given by
its polar components is pushed to Cartesian coordinates, where its
divergence is the trace of the Jacobian matrix; the result agrees with
$\frac1\rho\sum_i\partial_i(\rho X_i)$, and not with
$\sum_i\partial_iX_i$. Same for the gradient and the Laplacian.

In [4]:
polar = lambda u: torch.stack([u[0] * torch.cos(u[1]), u[0] * torch.sin(u[1])])
inv_polar = lambda x: torch.stack([x.norm(), torch.atan2(x[1], x[0])])
Xpol = lambda u: torch.stack([u[0] ** 2 * torch.cos(u[1]), torch.sin(2 * u[1]) / u[0] + u[0]])      # (X_r, X_theta)
# the same field in Cartesian coordinates: push forward by the polar map
Xcart = lambda x: jacrev(polar)(inv_polar(x)) @ Xpol(inv_polar(x))
u0 = T([1.3, 0.7])
rho = lambda u: u[0]
div_formula = torch.trace(jacrev(lambda u: rho(u) * Xpol(u))(u0)) / rho(u0)
div_cart = torch.trace(jacrev(Xcart)(polar(u0)))
naive = torch.trace(jacrev(Xpol)(u0))
assert torch.allclose(div_formula, div_cart) and abs(naive - div_cart) > 0.1
print(f"div X = {div_cart.item():.6f} (Cartesian) = {div_formula.item():.6f} (formula with rho = r);  sum of d_i X_i = {naive.item():.6f}")

# with the forms: d(i_X mu) = div_mu X mu, mu = r dr ^ dtheta
e2 = [dx(0, 2), dx(1, 2)]
mu_pol = lambda u: u[0] * wedge(e2[0], e2[1])
lhs = d(lambda u: interior(Xpol(u), mu_pol(u)))(u0)
assert torch.allclose(lhs, div_formula * mu_pol(u0))

# rule (3): div_{h mu} X = div_mu X + dh(X)/h
hfun = lambda u: 2.0 + torch.sin(u[0]) * u[1] ** 2
lhs = d(lambda u: interior(Xpol(u), hfun(u) * mu_pol(u)))(u0)[0, 1] / (hfun(u0) * u0[0])
assert torch.allclose(lhs, div_formula + torch.func.grad(hfun)(u0) @ Xpol(u0) / hfun(u0))

# gradient and Laplacian in polar coordinates
g = lambda x: torch.exp(0.3 * x[0]) * x[1] + x[0] ** 2 * x[1]
gp = lambda u: g(polar(u))
Gram = torch.diag(T([1.0, u0[0] ** 2]))
grad_polar = torch.linalg.inv(Gram) @ torch.func.grad(gp)(u0)
assert torch.allclose(jacrev(polar)(u0) @ grad_polar, torch.func.grad(g)(polar(u0)))
lap_cart = torch.trace(hessian(g)(polar(u0)))
Hp = hessian(gp)(u0)
lap_polar = torch.func.grad(gp)(u0)[0] / u0[0] + Hp[0, 0] + Hp[1, 1] / u0[0] ** 2
assert torch.allclose(lap_cart, lap_polar)
harm = lambda u: u[0] ** 3 * torch.cos(3 * u[1])
Hh = hessian(harm)(u0)
assert abs(torch.func.grad(harm)(u0)[0] / u0[0] + Hh[0, 0] + Hh[1, 1] / u0[0] ** 2) < 1e-12
print(f"Laplacian: {lap_cart.item():.6f} in Cartesian and in polar coordinates;  r^3 cos(3 theta) is harmonic")

# Exercise 24.2 (2): r^-3 E has no divergence
assert abs(div_of(lambda q: q / q.norm() ** 3)(p)) < 1e-12

div X = 3.244372 (Cartesian) = 3.244372 (formula with rho = r);  sum of d_i X_i = 2.250078


Laplacian: 1.776536 in Cartesian and in polar coordinates;  r^3 cos(3 theta) is harmonic


## 4. The divergence theorem (Theorem 24.10, Corollary 24.11)

Classical side: $\int_{\partial D}\langle X,\nu\rangle\,d\sigma$ with the
unit normal and the area density $\sqrt{\det G}$. Forms side: the integral
of the flux form. Volume side: $\int_D\operatorname{div}X$.

In [5]:
sph = lambda u: torch.stack([torch.sin(u[0]) * torch.cos(u[1]), torch.sin(u[0]) * torch.sin(u[1]), torch.cos(u[0])])
scale = T([1.5, 0.8, 1.1])
surf = lambda u: sph(u) * scale                                  # an ellipsoid
solid = lambda v: v[0] * surf(v[1:])                             # its inside, (t, theta, phi)


def area_density(param):
    def rho(u):
        Jp = jacrev(param)(u)
        return torch.sqrt(torch.linalg.det(Jp.T @ Jp))
    return rho


def unit_normal(param):
    # for a surface in R^3: the normalized cross product of the two coordinate vectors
    def nu(u):
        Jp = jacrev(param)(u)
        nrm = torch.linalg.cross(Jp[:, 0], Jp[:, 1])
        return nrm / nrm.norm()
    return nu


rules_s = [gauss(0, math.pi, 40), periodic(64)]
rules_v = [gauss(0, 1, 16), gauss(0, math.pi, 40), periodic(64)]
vol_side = integrate(lambda v: torch.trace(jacrev(X)(solid(v))) * torch.linalg.det(jacrev(solid)(v)), rules_v)
classical = integrate(lambda u: X(surf(u)) @ unit_normal(surf)(u) * area_density(surf)(u), rules_s)
forms = integrate(top(flux_form, surf), rules_s)
assert abs(vol_side - classical) < 1e-9 and abs(classical - forms) < 1e-9
print(f"ellipsoid: int div X = {vol_side.item():.8f},  flux with the unit normal = {classical.item():.8f},  integral of the flux form = {forms.item():.8f}")

# the normal computed from the parametrization is the outward one
u1 = T([0.9, 2.2])
assert unit_normal(surf)(u1) @ surf(u1) > 0

# Exercise 24.3
cube = lambda q: q ** 3
flux = integrate(lambda u: cube(sph(u)) @ sph(u) * area_density(sph)(u), [gauss(0, math.pi, 40), periodic(64)])
x4 = integrate(lambda u: sph(u)[0] ** 4 * area_density(sph)(u), [gauss(0, math.pi, 40), periodic(64)])
assert abs(flux - 12 * math.pi / 5) < 1e-10 and abs(x4 - 4 * math.pi / 5) < 1e-10
print(f"X = (x^3, y^3, z^3) on the unit ball: flux = {flux.item():.8f} = 12 pi / 5")

# Corollary 24.11: flux out of a small sphere, divided by the volume of the ball
# (for the field X above the quotient is exact for every r, because div X = 1 + xy is harmonic; take another field)
Xm = lambda q: torch.stack([torch.sin(q[0]) * q[1] ** 2, torch.exp(0.5 * q[1]) * q[2], q[2] ** 3 * q[0]])
div_p = torch.trace(jacrev(Xm)(p))
errs = []
for r in (0.4, 0.2, 0.1):
    small = lambda u, r=r: p + r * sph(u)
    fl = integrate(lambda u: Xm(small(u)) @ sph(u) * r ** 2 * torch.sin(u[0]), [gauss(0, math.pi, 40), periodic(64)])
    errs.append(abs(fl / (4 / 3 * math.pi * r ** 3) - div_p).item())
    print(f"r = {r}: flux / volume = {(fl / (4 / 3 * math.pi * r ** 3)).item():+.6f},  div X(p) = {div_p.item():+.6f}")
assert errs[1] < errs[0] / 3 and errs[2] < errs[1] / 3

ellipsoid: int div X = 5.52920307,  flux with the unit normal = 5.52920307,  integral of the flux form = 5.52920307
X = (x^3, y^3, z^3) on the unit ball: flux = 7.53982237 = 12 pi / 5
r = 0.4: flux / volume = +1.804244,  div X(p) = +1.732992
r = 0.2: flux / volume = +1.750835,  div X(p) = +1.732992
r = 0.1: flux / volume = +1.737455,  div X(p) = +1.732992


## 5. The Kelvin–Stokes theorem (Theorem 24.12) and Green's identities

On the piece $z = x^2 + y^2$, $x^2 + y^2\leq1$ of a paraboloid, with the
normal on the upper side. Classical side: unit normal, area, unit tangent
and length.

In [6]:
cap = lambda v: torch.stack([v[0] * torch.cos(v[1]), v[0] * torch.sin(v[1]), v[0] ** 2])
assert unit_normal(cap)(T([0.5, 1.0]))[2] > 0                    # the coordinates (t, angle) give the upper normal
lhs = integrate(lambda v: curl_of(X)(cap(v)) @ unit_normal(cap)(v) * area_density(cap)(v), [gauss(0, 1, 30), periodic(64)])
rim = lambda a: cap(torch.cat([torch.ones(1), a]))


def circulation(u):
    c, dc = jvp(rim, (u,), (torch.ones(1),))
    return X(c) @ dc                                              # <X, tau> ds = <X, c'> da


rhs = integrate(circulation, [periodic(64)])
assert abs(lhs - rhs) < 1e-10
print(f"flux of curl X through the surface = {lhs.item():+.10f},  circulation along its boundary = {rhs.item():+.10f}")

# Exercise 24.4: X = (-y, x, 0), upper half sphere and flat disc
half = [gauss(0, math.pi / 2, 30), periodic(32)]
a_ = integrate(lambda u: curl_of(Ye)(sph(u)) @ sph(u) * area_density(sph)(u), half)
disc3 = lambda v: torch.stack([v[0] * torch.cos(v[1]), v[0] * torch.sin(v[1]), 0 * v[0]])
b_ = integrate(lambda v: curl_of(Ye)(disc3(v))[2] * v[0], [gauss(0, 1, 10), periodic(32)])
assert abs(a_ - 2 * math.pi) < 1e-10 and abs(b_ - 2 * math.pi) < 1e-10

# Green's first identity on the unit disc of the plane
fg = lambda x: torch.sin(x[0]) * x[1] + x[0] ** 2
gg = lambda x: torch.exp(0.5 * x[1]) * x[0] + x[1] ** 3
disc = lambda v: v[0] * torch.stack([torch.cos(v[1]), torch.sin(v[1])])
inside = integrate(lambda v: (fg(disc(v)) * torch.trace(hessian(gg)(disc(v)))
                              + torch.func.grad(fg)(disc(v)) @ torch.func.grad(gg)(disc(v))) * v[0],
                   [gauss(0, 1, 30), periodic(64)])
circle = lambda a: torch.stack([torch.cos(a[0]), torch.sin(a[0])])
bdry = integrate(lambda a: fg(circle(a)) * (torch.func.grad(gg)(circle(a)) @ circle(a)), [periodic(64)])
assert abs(inside - bdry) < 1e-10
print(f"Green: int (f Lap g + <grad f, grad g>) = {inside.item():.10f} = int f d_nu g = {bdry.item():.10f}")

flux of curl X through the surface = -3.1415926536,  circulation along its boundary = -3.1415926536
Green: int (f Lap g + <grad f, grad g>) = 0.7530566981 = int f d_nu g = 0.7530566981


## 6. The Laplacian with another volume

For the standard Gaussian density $p$,
$\Delta_pf = \Delta f - \langle x, \operatorname{grad}f\rangle$. Its
symmetry in $L^2(p)$, and the eigenfunctions of Exercise 24.5, are checked
with Gauss–Hermite quadrature, which is exact for polynomials.

In [7]:
xh, wh = np.polynomial.hermite_e.hermegauss(12)                   # weight exp(-x^2/2)
xh, wh = T(xh), T(wh) / math.sqrt(2 * math.pi)
grid = torch.cartesian_prod(xh, xh)
wgrid = torch.cartesian_prod(wh, wh).prod(1)
E = lambda fn: (vmap(fn)(grid) * wgrid).sum()                     # mean under the standard Gaussian of R^2

lap_p = lambda fn: (lambda x: torch.trace(hessian(fn)(x)) - x @ torch.func.grad(fn)(x))
x1 = lambda x: x[0]
h2 = lambda x: x[1] ** 2 - 1
x12 = lambda x: x[0] * x[1]
pt = T([0.4, -1.7])
assert torch.allclose(lap_p(x1)(pt), -x1(pt)) and torch.allclose(lap_p(h2)(pt), -2 * h2(pt))
assert torch.allclose(lap_p(x12)(pt), -2 * x12(pt))

fa = lambda x: x[0] ** 3 + x[0] * x[1] - x[1] ** 2
fb = lambda x: x[0] ** 2 * x[1] + 2 * x[1] ** 3 + x[0]
left = E(lambda x: fa(x) * lap_p(fb)(x))
mid = -E(lambda x: torch.func.grad(fa)(x) @ torch.func.grad(fb)(x))
right = E(lambda x: lap_p(fa)(x) * fb(x))
assert torch.allclose(left, mid) and torch.allclose(mid, right)
assert abs(E(lambda x: x1(x) * h2(x))) < 1e-12
print(f"E[f Lap_p g] = {left.item():.8f} = -E<grad f, grad g> = {mid.item():.8f} = E[g Lap_p f] = {right.item():.8f}")

# div_p X = div X + <grad ln p, X> has mean 0 (Stein's identity as a divergence)
Xg = lambda x: torch.stack([x[0] ** 2 * x[1] + x[1], x[0] - x[1] ** 3])
assert abs(E(lambda x: torch.trace(jacrev(Xg)(x)) - x @ Xg(x))) < 1e-12

E[f Lap_p g] = -3.00000000 = -E<grad f, grad g> = -3.00000000 = E[g Lap_p f] = -3.00000000


## 7. Hutchinson's estimator (Proposition 24.16)

The mean and the variance for signs are computed exactly, by listing the
$2^n$ sign vectors. For Gaussians they are estimated. Then the divergence
of a small network is estimated with Jacobian–vector products.

In [8]:
gen = torch.Generator().manual_seed(4)
n = 6
Jm = torch.randn(n, n, generator=gen)
S = 0.5 * (Jm + Jm.T)
signs = T(list(itertools.product([-1.0, 1.0], repeat=n)))
vals = torch.einsum("ki,ij,kj->k", signs, Jm, signs)
off = S - torch.diag(torch.diag(S))
assert torch.allclose(vals.mean(), torch.trace(Jm))
assert torch.allclose(vals.var(unbiased=False), 2 * (off ** 2).sum())
print(f"signs:     mean {vals.mean().item():.6f} = tr J,  variance {vals.var(unbiased=False).item():.6f} = 2 sum_(i != j) S_ij^2")

N = 400_000
U = torch.randn(N, n, generator=gen)
gv = torch.einsum("ki,ij,kj->k", U, Jm, U)
se_mean = gv.std() / math.sqrt(N)
assert abs(gv.mean() - torch.trace(Jm)) < 4 * se_mean
target = 2 * (S ** 2).sum()
assert abs(gv.var() - target) < 0.03 * target
print(f"Gaussians: mean {gv.mean().item():.4f},  variance {gv.var().item():.4f},  2 |S|^2 = {target.item():.4f}")

# Exercise 24.7
Jd = torch.diag(T([1.0, -2.0, 0.5]))
s3 = T(list(itertools.product([-1.0, 1.0], repeat=3)))
assert torch.all(torch.einsum("ki,ij,kj->k", s3, Jd, s3) == torch.trace(Jd))
Ja = Jm - Jm.T
assert torch.einsum("ki,ij,kj->k", signs, Ja, signs).abs().max() < 1e-12
Jex = T([[1.0, 2.0], [0.0, 3.0]])
s2 = T(list(itertools.product([1.0, -1.0], repeat=2)))
v2 = torch.einsum("ki,ij,kj->k", s2, Jex, s2)
assert v2.tolist() == [6.0, 2.0, 2.0, 6.0] and v2.mean() == 4 and v2.var(unbiased=False) == 4

# the divergence of a network, from Jacobian-vector products
n = 20
W1, b1 = torch.randn(64, n, generator=gen) / math.sqrt(n), torch.randn(64, generator=gen)
W2 = torch.randn(n, 64, generator=gen) / 8
net = lambda x: W2 @ torch.tanh(W1 @ x + b1)
x0 = torch.randn(n, generator=gen)
exact = torch.trace(jacrev(net)(x0))


def probe(u):
    _, Ju = jvp(net, (x0,), (u,))
    return u @ Ju


K_probes = 2000
rad = torch.randint(0, 2, (K_probes, n), generator=gen).double() * 2 - 1
est = vmap(probe)(rad)
assert abs(est.mean() - exact) < 4 * est.std() / math.sqrt(K_probes)
print(f"divergence of a network in dimension {n}: exact {exact.item():.4f},  {K_probes} probes {est.mean().item():.4f} "
      f"+- {(est.std() / math.sqrt(K_probes)).item():.4f}")

# the Laplacian of a scalar network from Hessian-vector products
w3 = torch.randn(64, generator=gen) / 8
scalar = lambda x: w3 @ torch.tanh(W1 @ x + b1)
lap_exact = torch.trace(hessian(scalar)(x0))


def hvp_probe(u):
    _, Hu = jvp(torch.func.grad(scalar), (x0,), (u,))
    return u @ Hu


est = vmap(hvp_probe)(rad)
assert abs(est.mean() - lap_exact) < 4 * est.std() / math.sqrt(K_probes)
print(f"Laplacian of a scalar network: exact {lap_exact.item():.4f},  estimate {est.mean().item():.4f}")

signs:     mean 3.777923 = tr J,  variance 43.325879 = 2 sum_(i != j) S_ij^2


Gaussians: mean 3.7993,  variance 62.1850,  2 |S|^2 = 61.9525


divergence of a network in dimension 20: exact 0.0721,  2000 probes 0.0167 +- 0.0633
Laplacian of a scalar network: exact 1.4778,  estimate 1.4726


## Summary

* $i_X\mathrm{vol} = \star(RX)$ for any inner product.
* The operators defined through forms are the classical gradient, curl and
  divergence; $d\circ d = 0$ gives the two identities, and the operator $K$
  of the Poincaré lemma gives the potentials.
* In polar coordinates the divergence is $\frac1r\partial_r(rX_r) +
  \partial_\theta X_\theta$, not the sum of the partial derivatives of the
  components.
* The divergence theorem and the Kelvin–Stokes theorem hold with the
  classical integrands, and agree with the integrals of the flux form and
  the work form.
* $\Delta_p$ is symmetric in $L^2(p)$, with the Hermite polynomials as
  eigenfunctions for the Gaussian.
* Hutchinson's estimator is unbiased, with variance $2\sum_{i\neq j}S_{ij}^2$
  for signs and $2\|S\|^2$ for Gaussians.